In [1]:
import pandas as pd
import numpy as np

print("Kütüphaneler başarıyla yüklendi.")

Kütüphaneler başarıyla yüklendi.


In [2]:
data = {
    "Alternatif": ["A1", "A2", "A3", "A4", "A5"],
    "Urun": [
        "Wayona USB Cable",
        "Ambrane USB Cable",
        "Sounce USB Cable",
        "boAt USB Cable",
        "Portronics USB Cable"
    ],
    "Rating": [4.1, 4.0, 3.9, 4.2, 4.2],
    "Rating_Count": [24269, 43994, 7928, 94363, 16905],
    "Discount": [64, 43, 90, 53, 61],
    "Discounted_Price": [399, 199, 199, 329, 154],
    "Actual_Price": [1099, 349, 1899, 699, 399]
}

df = pd.DataFrame(data)

df

,Alternatif,Urun,Rating,Rating_Count,Discount,Discounted_Price,Actual_Price
0,A1,Wayona USB Cable,4.1,24269,64,399,1099
1,A2,Ambrane USB Cable,4.0,43994,43,199,349
2,A3,Sounce USB Cable,3.9,7928,90,199,1899
3,A4,boAt USB Cable,4.2,94363,53,329,699
4,A5,Portronics USB Cable,4.2,16905,61,154,399


In [3]:
# AHP sonucunda elde edilen kriter ağırlıkları

weights = np.array([
    0.48,  # Rating
    0.26,  # Rating Count
    0.13,  # Discount
    0.08,  # Discounted Price
    0.05   # Actual Price
])

print("AHP Kriter Ağırlıkları")
print(weights)

AHP Kriter Ağırlıkları
[0.48 0.26 0.13 0.08 0.05]


In [4]:
# TOPSIS analizinde kullanılacak kriter sütunları

criteria = [
    "Rating",
    "Rating_Count",
    "Discount",
    "Discounted_Price",
    "Actual_Price"
]

# Karar matrisi
X = df[criteria].values.astype(float)

# Vektör normalizasyonu
norms = np.sqrt((X ** 2).sum(axis=0))
normalized_matrix = X / norms

normalized_df = pd.DataFrame(
    normalized_matrix,
    columns=criteria,
    index=df["Alternatif"]
)

print("Normalize Karar Matrisi")
normalized_df

Normalize Karar Matrisi


,Rating,Rating_Count,Discount,Discounted_Price,Actual_Price
Alternatif,,,,,
A1,0.449222,0.223628,0.446180,0.655633,0.465094
A2,0.438266,0.405385,0.299777,0.326995,0.147696
A3,0.427309,0.073053,0.627441,0.326995,0.803651
A4,0.460179,0.869513,0.369493,0.540610,0.295815
A5,0.460179,0.155772,0.425265,0.253051,0.168856


In [5]:
# AHP ağırlıkları ile normalize karar matrisinin çarpılması

weighted_matrix = normalized_matrix * weights

weighted_df = pd.DataFrame(
    weighted_matrix,
    columns=criteria,
    index=df["Alternatif"]
)

print("Ağırlıklı Normalize Karar Matrisi")
weighted_df

Ağırlıklı Normalize Karar Matrisi


,Rating,Rating_Count,Discount,Discounted_Price,Actual_Price
Alternatif,,,,,
A1,0.215627,0.058143,0.058003,0.052451,0.023255
A2,0.210368,0.105400,0.038971,0.026160,0.007385
A3,0.205108,0.018994,0.081567,0.026160,0.040183
A4,0.220886,0.226073,0.048034,0.043249,0.014791
A5,0.220886,0.040501,0.055284,0.020244,0.008443


In [6]:
# Fayda kriterleri: Rating, Rating Count, Discount
# Maliyet kriterleri: Discounted Price, Actual Price

benefit_criteria = [0, 1, 2]
cost_criteria = [3, 4]

positive_ideal = np.zeros(weighted_matrix.shape[1])
negative_ideal = np.zeros(weighted_matrix.shape[1])

for i in range(weighted_matrix.shape[1]):
    if i in benefit_criteria:
        positive_ideal[i] = weighted_matrix[:, i].max()
        negative_ideal[i] = weighted_matrix[:, i].min()
    else:
        positive_ideal[i] = weighted_matrix[:, i].min()
        negative_ideal[i] = weighted_matrix[:, i].max()

# Uzaklık hesapları
distance_positive = np.sqrt(((weighted_matrix - positive_ideal) ** 2).sum(axis=1))
distance_negative = np.sqrt(((weighted_matrix - negative_ideal) ** 2).sum(axis=1))

# TOPSIS skoru
topsis_score = distance_negative / (distance_positive + distance_negative)

results = df[["Alternatif", "Urun"]].copy()
results["TOPSIS_Skoru"] = topsis_score
results["Siralama"] = results["TOPSIS_Skoru"].rank(ascending=False).astype(int)

results = results.sort_values("Siralama")

print("TOPSIS Sonuçları")
results

TOPSIS Sonuçları


,Alternatif,Urun,TOPSIS_Skoru,Siralama
3,A4,boAt USB Cable,0.835294,1
1,A2,Ambrane USB Cable,0.428135,2
4,A5,Portronics USB Cable,0.226793,3
0,A1,Wayona USB Cable,0.216349,4
2,A3,Sounce USB Cable,0.192234,5
